# Практическое занятие 8
## Статистические методы обнаружения аномалий в кибербезопасности

Продолжительность: 80 минут. Работайте последовательно. Поле `scenario` используется только для учебной проверки и не включается в расчёт статистических показателей.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as sts
from sklearn.covariance import MinCovDet
%config InlineBackend.figure_format = 'retina'

### Загрузка данных
Разместите `cyber_statistical_anomalies.csv` рядом с ноутбуком. В Google Colab загрузите CSV в рабочую область с тем же именем.

In [ ]:
DATA_PATH = 'cyber_statistical_anomalies.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df.head()

## Задание 1. Подготовка данных и постановка задачи

1. Выведите размер, типы столбцов, число пропусков и полных дубликатов.
2. Проверьте распределение учебных сценариев.
3. Укажите, какие внедрённые сценарии являются точечными, коллективными или многомерными.
4. Объясните, почему `scenario` нельзя использовать как входной признак.
5. Зафиксируйте базовый интервал `0:119` для многомерного профиля.

In [ ]:
# аудит данных
audit = {
    'Размер': None,
    'Пропуски': None,
    'Дубликаты': None,
}
audit

In [ ]:
# распределение сценариев
scenario_counts = None
scenario_counts

**Краткий вывод:** [укажите виды аномалий, роль базового интервала и назначение учебной метки]

## Задание 2. Одномерные пороговые методы

1. Для `failed_logins` вычислите Z-score и отметьте значения `z > 3`.
2. Для `bytes_out_mb` вычислите границы IQR с коэффициентом 1,5.
3. Для того же признака вычислите модифицированный Z-score с порогом `|M| > 3.5`.
4. Рассчитайте верхний квантиль 0,995.
5. Сравните списки событий и объясните ложные срабатывания IQR/MAD на правосторонне асимметричном распределении.
6. Постройте график Z-score и boxplot для исходящего трафика.

In [ ]:
# Z-score для failed_logins
feature = 'failed_logins'
mean_value = ...
std_value = ...
df['failed_logins_z'] = ...
df['z_anomaly'] = ...
df.loc[df['z_anomaly'], ['timestamp', feature, 'failed_logins_z', 'scenario']]

In [ ]:
# IQR, MAD и квантильный порог для bytes_out_mb
feature = 'bytes_out_mb'
q1 = ...
q3 = ...
iqr = ...
lower_limit = ...
upper_limit = ...
df['iqr_anomaly'] = ...

median_value = ...
mad = ...
if mad == 0:
    raise ValueError('MAD равен нулю')
df['modified_z'] = ...
df['mad_anomaly'] = ...

quantile_level = 0.995
quantile_limit = ...
df['quantile_anomaly'] = ...

In [ ]:
# сводная таблица количества найденных событий и графики
comparison = pd.Series({
    'Z-score': None, 'IQR': None, 'MAD': None, 'Квантиль': None
})
comparison

**Краткий вывод:** [какие методы нашли внедрённые сценарии; почему статистический выброс не равен атаке]

## Задание 3. Критерий Граббса

1. Реализуйте функцию из лекции для `session_duration_min` при `α = 0,05`.
2. Сформулируйте H0 и H1.
3. Сопоставьте G и Gcrit, укажите найденное наблюдение.
4. Объясните, почему тест не следует многократно повторять без поправки.

In [ ]:
# реализуйте критерий Граббса по формулам лекции
def grubbs_test(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    n = len(values)
    # mean, std, максимальное отклонение, G, t_critical, G_critical
    return {}

result = grubbs_test(df['session_duration_min'], alpha=0.05)
result

**Решение по гипотезе:** [отвергнуть или не отвергнуть H0 и почему]

## Задание 4. Многомерные аномалии

1. На базовом интервале `0:119` оцените средний вектор и ковариационную матрицу для `connections_per_min` и `unique_destinations`.
2. Рассчитайте квадрат расстояния Махаланобиса для всех наблюдений.
3. Используйте порог χ² уровня 0,99 при двух степенях свободы.
4. Повторите расчёт с `MinCovDet(random_state=42)`.
5. Сравните найденные события и объясните назначение робастной оценки.
6. Постройте диаграмму рассеяния и выделите аномалии классического метода.

In [ ]:
# классическое расстояние Махаланобиса
features = ['connections_per_min', 'unique_destinations']
reference_data = df.loc[0:119, features]
mean_vector = ...
covariance_matrix = ...
inverse_covariance = ...
differences = ...
df['mahalanobis_d2'] = ...
mahalanobis_limit = ...
df['mahalanobis_anomaly'] = ...

In [ ]:
# робастное расстояние Махаланобиса
robust_model = MinCovDet(random_state=42)
# fit, mahalanobis, порог
df['robust_mahalanobis_d2'] = ...
df['robust_mahalanobis_anomaly'] = ...

In [ ]:
# сравнение результатов и диаграмма рассеяния
df.loc[df['mahalanobis_anomaly'], ['timestamp', *features, 'mahalanobis_d2', 'scenario']]

**Краткий вывод:** [какое необычное сочетание признаков обнаружено; чем различаются классическая и робастная оценки]

## Задание 5. Контрольные карты Шухарта и записка SOC

1. Разбейте `requests_per_min` на 8-часовые окна.
2. Рассчитайте по окнам среднее и стандартное отклонение.
3. Для X-bar и R-карты вычислите CL, UCL и LCL по правилу 3σ. Для R-карты ограничьте LCL снизу нулём.
4. Найдите окна выше UCL и постройте две контрольные карты.
5. Подготовьте аналитическую записку SOC из 5–7 предложений.

In [ ]:
# 8-часовые окна и контрольные границы
df['window'] = df['timestamp'].dt.floor('8h')
grouped = ...
# CL_mean, sigma_mean, UCL_mean, LCL_mean
# CL_std, sigma_std, UCL_std, LCL_std
grouped.head()

In [ ]:
# найдите аномальные окна и постройте X-bar и R-карты

### Записка SOC

**Период анализа:** [период]

**Методы и пороги:** [перечень]

**Выявленные события и окна:** [время, признак, значение]

**Интерпретация:** [что требует проверки; не приравнивайте аномалию к атаке]

**Рекомендации:** [какой контекст и журналы проверить]

**Ограничения:** [предпосылки и ограничения методов]